In [38]:
from pathlib import Path
import sys
import os
import re
import mne
import pandas as pd
import numpy as np
from scipy import signal
import pywt
sys.path.append('..')
#from libML.data_loading import find_bids_emg_files, load_emg_bids

In [2]:
p = Path.cwd()
repo_root = None
while True:
    if (p / "data").exists() and (p / "notebooks").exists():
        repo_root = p
        break
    if p == p.parent:
        raise FileNotFoundError("Could not find project root containing 'data' and 'notebooks'")
    p = p.parent
data_path = repo_root / "data" / "raw" / "sub-01" / "emg"

In [3]:
repo_root

WindowsPath('c:/Users/miski/Desktop/Neuro-X/N-Pulse/BMI-SOFT-Signal_Processing_ML')

In [4]:
def find_bids_emg_files(bids_root, subject, session=None, task=None):
    """
    Return list of (edf_path, events_path, json_path) for one subject/session/task.
    """
    subj_dir = os.path.join(bids_root, "data")
    subj_dir = os.path.join(subj_dir, "raw")
    subj_dir = os.path.join(subj_dir, f"sub-{subject}")
    if session:
        subj_dir = os.path.join(subj_dir, f"ses-{session}")
    if task:
        subj_dir = os.path.join(subj_dir, f"task-{task}")

    emg_dir = os.path.join(subj_dir, "emg")
    print(emg_dir)
    if not os.path.exists(emg_dir):
        raise FileNotFoundError(f"No EMG folder for subject {subject}, session {session}")

    pattern = f"sub-{subject}"
    if session:
        pattern += f"_ses-{session}"
    if task:
        pattern += f"_task-{task}"
    pattern += ".*_emg\\.edf$"

    files = [f for f in os.listdir(emg_dir) if re.match(pattern, f)]
    return [os.path.join(emg_dir, f) for f in files]


In [5]:
find_bids_emg_files(repo_root, subject="01") #, task="Cyl_acq-cometa"

c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg


['c:\\Users\\miski\\Desktop\\Neuro-X\\N-Pulse\\BMI-SOFT-Signal_Processing_ML\\data\\raw\\sub-01\\emg\\sub-01_task-Cyl_acq-cometa_emg.edf']

In [6]:
def load_emg_bids(bids_root, subject, task=None, session=None, label_column="event_label"):
    """
    Load EMG data and labels from a BIDS-compliant directory.
    Returns: X (samples x channels), y (labels)
    """
    edf_files = find_bids_emg_files(bids_root, subject, session) #task not input because in our example not specified in the path
    if len(edf_files) == 0:
        raise FileNotFoundError(f"No EMG EDF found for task={task}")

    edf_path = edf_files[0]  # assuming one run
    base_prefix = edf_path.replace("_emg.edf", "")
    events_path = base_prefix + "_events.tsv"

    # --- Load EMG signal ---
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    emg_data = raw.get_data().T  # shape: (n_samples, n_channels)
    sfreq = raw.info["sfreq"]

    # --- Load events ---
    #events_df = pd.read_csv(events_path, sep="\t")

    #if label_column not in events_df.columns:
    #    raise ValueError(f"Missing {label_column} in events.tsv")

    # Align EMG and events (simple approach: expand labels per time window)
    #y = events_df[label_column].to_numpy()
    y=None
    return emg_data, y, sfreq

In [7]:
emg_data, y, sfreq = load_emg_bids(repo_root, subject="01", task="Cyl_acq-cometa")

c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg
Extracting EDF parameters from c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg\sub-01_task-Cyl_acq-cometa_emg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12499  =      0.000 ...   124.990 secs...


C:\Users\miski\AppData\Local\Temp\ipykernel_952\3000727891.py:15: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_path, preload=True)


In [8]:
emg_data.shape

(12500, 8)

In [9]:

def find_bids_emg_files(bids_root, subject, session=None, task=None, run=None):
    """
    Return list of (xdf_path, events_path, json_path) for one subject/session/task.
    """
    subj_dir = os.path.join(bids_root, "data")
    subj_dir = os.path.join(subj_dir, "raw")
    subj_dir = os.path.join(subj_dir, f"sub-{subject}")
    if session:
        subj_dir = os.path.join(subj_dir, f"ses-{session}")
    #if task:
    #    subj_dir = os.path.join(subj_dir, f"task-{task}")
    #if run:
    #    subj_dir = os.path.join(subj_dir, f"run-{run}")
    emg_dir = os.path.join(subj_dir, "emg")
    #print(emg_dir)
    if not os.path.exists(emg_dir):
        raise FileNotFoundError(f"No EMG folder for subject {subject}, session {session}")

    pattern = f"sub-{subject}"
    if session:
        pattern += f"_ses-{session}"
    if task:
        pattern += f"_task-{task}"
    if run:
        pattern += f"_run-{run}"
    pattern += ".xdf"
    
    files = [f for f in os.listdir(emg_dir) if re.match(pattern, f)]
    return [os.path.join(emg_dir, f) for f in files]

def load_emg_bids(bids_root, subject, session=None, task=None, run=None, label_column="event_label"):
    """
    Load EMG data and labels from a BIDS-compliant directory.
    Returns: X (samples x channels), y (labels)
    """
    xdf_files = find_bids_emg_files(bids_root, subject, session, task, run)
    if len(xdf_files) == 0:
        raise FileNotFoundError(f"No EMG EDF found for task={task}")

    xdf_path = xdf_files[0]  # assuming one run
    base_prefix = xdf_path.replace("_emg.xdf", "")
    events_path = base_prefix + "_events.tsv"

    # --- Load EMG signal ---
    """ raw = mne.io.read_raw_edf(xdf_path, preload=True)
    emg_data = raw.get_data().T  # shape: (n_samples, n_channels)
    sfreq = raw.info["sfreq"] """
    streams, header = pyxdf.load_xdf(xdf_path)
    """ data = streams[0]["time_series"].T
    sfreq = float(streams[0]["info"]["nominal_srate"][0])
    info = mne.create_info(3, sfreq, ["eeg", "eeg", "stim"])
    raw = mne.io.RawArray(data, info)
    raw.plot(scalings=dict(eeg=100e-6), duration=1, start=14) """

    # --- Load events ---
    #events_df = pd.read_csv(events_path, sep="\t")

    #if label_column not in events_df.columns:
    #    raise ValueError(f"Missing {label_column} in events.tsv")

    # Align EMG and events (simple approach: expand labels per time window)
    #y = events_df[label_column].to_numpy()
    y=None
    return streams, header

def get_emg_channels(streams):
    """
    Extract EMG channels from the list of streams.
    
    Args:
        streams: List of data streams loaded from XDF file.
    """
    trigger_labels = streams[0]
    all_channels = streams[1]

    emg_data = all_channels['time_series']
    timestamps = all_channels['time_stamps']

    try:
        channels = all_channels['info']['desc'][0]['channels'][0]['channel']
        channel_names = [ch['label'][0] for ch in channels]
    except KeyError:
        channel_names = [f'Channel_{i+1}' for i in range(emg_data.shape[1])]
    
    # Extract channel names from the info structure
    try:
        channels = all_channels['info']['desc'][0]['channels'][0]['channel']
        channel_names = [ch['label'][0] for ch in channels]
    except KeyError as e:
        print(f"Error extracting channel names: {e}")
        channel_names = [f'Channel_{i+1}' for i in range(emg_data.shape[1])]
    
    # Determine data orientation
    if emg_data.shape[1] == len(channel_names):
        data_for_plotting = emg_data
    else:
        data_for_plotting = emg_data.T
    
    # Get the last 6 channels EXCLUDING the very last one (trigger channel)
    last_6_indices = range(len(channel_names)-7, len(channel_names)-1)
    last_6_names = [channel_names[i] for i in last_6_indices]
    
    # Extract the time series data for these 6 channels
    last_6_time_series = data_for_plotting[:, last_6_indices]
    last_6_timestamps = timestamps[:]
    
    # Return all the data
    emg_channels = {
        'channel_names': last_6_names,
        'time_series': last_6_time_series,
        'time_stamps': last_6_timestamps,
        'excluded_trigger': channel_names[-1]
    }

    return emg_channels

# Testing xdf loading and labelling

In [10]:
import pyxdf

In [11]:

def map_triggers_to_labels(emg_channels, trigger_labels):
    """
    Map trigger signals to action labels.
    It also deletes the data that was sampled before the first trigger.

    Args:
        emg_channels: EMG data channels (will be reformated)
        trigger_channel: Trigger channel (0 for each timestamp) but gives us length of data 
                         and corresponding timestamps
        trigger_labels: Trigger labels following the Data Acquisition Protocol

    Returns:
        train_labels: List of labels (format needed for training) for each timestamp after the first trigger
        emg_clean: Cleand EMG channels starting from the first trigger
    """
    emg_channels = emg_channels.copy()
    trigger_labels = trigger_labels.copy()

    # First step: Find the first trigger index and trim data preceding it
    emg_clean = clean_emg_from_index(emg_channels, trigger_labels)

    # Second step: Define Map that transforms triggers to labels
    # Create a copy of the time_series array to avoid modifying the original
    mapped_trigger_series = map_protocol_to_label(trigger_labels["time_series"].copy())
    
    # Create a new dictionary with mapped labels but keep the original structure
    mapped_trigger_labels = {
        "time_stamps": trigger_labels["time_stamps"],
        "time_series": mapped_trigger_series
    }

    # Third step: Create train_labels list --> label each timestamp
    emg_labeled = extend_labels(emg_clean, mapped_trigger_labels)

    return emg_labeled

def clean_emg_from_index(emg_channels, trigger_labels):
    """
    Find the index of the first trigger in the trigger channel.

    Args:
        emg_channels: EMG data channels
        trigger_labels: Trigger labels following the Data Acquisition Protocol

    Returns:
        emg_channels, trigger_channel: Filtered EMG and trigger channels starting from first trigger
    """
    ts = np.array(emg_channels["time_stamps"])
    if ts.size == 0:
        print('No timestamps in channels to filter.')
    else:
        first_label_ts = trigger_labels["time_stamps"][0]
        mask = ts >= first_label_ts
        removed = np.count_nonzero(~mask)
        # Update timestamps
        emg_channels["time_stamps"] = ts[mask].tolist()
        # Update corresponding time_series. Handle common orientations:
        emg_data = np.array(emg_channels["time_series"])
        # If rows correspond to timestamps (n_samples, n_channels)
        if emg_data.shape[0] == ts.size:
            emg_channels["time_series"] = emg_data[mask].tolist()
        # If columns correspond to timestamps (n_channels, n_samples)
        elif emg_data.ndim == 2 and emg_data.shape[1] == ts.size:
            emg_channels["time_series"] = emg_data[:, mask].tolist()
        else:
            # Fallback: try to filter rows; if that fails, leave emg_data as-is
            try:
                emg_channels["time_series"] = emg_data[mask].tolist()
            except Exception as e:
                print(f'Could not apply mask to time_series automatically: {e}')

    return emg_channels

def extend_labels(emg_channels, trigger_labels):
    """
    Extend the data acquisition protocol labels to every timestamp

    Args:
        emg_channels: EMG data channels
        trigger_labels: Trigger labels following the Data Acquisition Protocol

    Returns:
        emg_channels, trigger_channel: Filtered EMG and trigger channels starting from first trigger
    """
    # Expect trigger_labels to be streams[0] and emg_channels to be produced earlier
    trigger_ts = trigger_labels["time_stamps"]
    trigger_vals = trigger_labels["time_series"]
    # Flatten and decode bytes if necessary
    if trigger_vals.ndim > 1:
        trigger_vals = trigger_vals.flatten()
    trigger_vals = [v.decode() if isinstance(v, (bytes, bytearray)) else v for v in trigger_vals]

    emg_ts = np.array(emg_channels.get("time_stamps", []))

    # Prepare labels array aligned to emg_ts
    if trigger_ts.size == 0 or emg_ts.size == 0:
        # Nothing to map; create same-length None list
        emg_channels["labels"] = [None] * emg_ts.size
        print("No trigger timestamps or no EMG timestamps — created empty labels list")
    else:
        # Find for each emg timestamp the index of the latest trigger <= ts
        inds = np.searchsorted(trigger_ts, emg_ts, side="right") - 1
        labels = []
        for idx in inds:
            if idx < 0:
                # Timestamp occurs before the first trigger
                labels.append(None)
            else:
                labels.append(trigger_vals[idx])
        emg_channels["labels"] = labels

    return emg_channels

def map_protocol_to_label(trigger_labels):
    """
    Dummy version : Not taking into account initial hand state, only final state of the action

    For each protocol movement, label 8 degrees of freedom :
        1.  Thumb flexed/rest/extended  -->  0/1/2
        2.  Index flexed/rest/extended  -->  0/1/2
        3.  Middle flexed/rest/extended -->  0/1/2
        4.  Ring flexed/rest/extended   -->  0/1/2
        5.  Little flexed/rest/extended -->  0/1/2
        6.  Supination  Palm facing: Up/Side/Down  -->  0/1/2
        7.  Wrist angle Palm facing down then: Up(-90)/Straight(0)/Down(90)  -->  0/1/2
        8.  Thumb Abduction --> extended dorsal / rest / extended palmar --> 0/1/2
        9.  Strength --> not specified / low / medium / high --> 0/1/2/3
        10. Speed --> not specified / low / medium / high --> 0/1/2/3

    Label is encoded in a 10-digit value (int)
    Label = -1 is for non interesting data

    """
    # Convert to numpy array with safe dtype first
    if not isinstance(trigger_labels, np.ndarray):
        trigger_labels = np.array(trigger_labels, dtype=np.int64)
    else:
        trigger_labels = trigger_labels.astype(np.int64)
    
    new_labels = np.full_like(trigger_labels, -1, dtype=np.int64)
    
    for i in range(len(trigger_labels)):
        label = trigger_labels[i]

        # Special codes
        if label == 9701 or label == 9702:
            if i != 0:
                new_labels[i] = -1  # or new_labels[i-1] if you want to carry forward
            else:  # in case we start the recording in resting state
                new_labels[i] = -1
            continue
        if label in [8888, 9999, 8899]:
            new_labels[i] = -1
            continue

        # Get codes
        phase_label = label // 10000
        arm_label = (label - phase_label * 10000) // 1000  # don't care actually
        baseline_label = (label - phase_label * 10000 - arm_label * 1000) // 100
        movement_label = label - phase_label * 10000 - arm_label * 1000 - baseline_label * 100
        
        new_label = 0

        # No movements
        if phase_label not in [3, 4]:  # move and return
            new_labels[i] = -1  # not interesting for training
            continue

        # Disregarded movements
        if movement_label in []:
            new_labels[i] = -1
            continue

        # Movements
        if phase_label in [3, 4]:
            # Use proper base-10 encoding instead of floating point exponents
            # Each digit gets its own place value (10^0, 10^1, 10^2, etc.)
            
            # Thumb flex/rest/ext --> 0/1/2 (1st digit)
            if movement_label in [3, 4, 5, 6, 10, 15, 16, 22]:  # flexed
                digit = 0
            elif movement_label in [7, 8, 11, 12, 13, 14, 17, 18, 19, 20, 21, 23, 24, 25, 26]:  # rest
                digit = 1
            else:  # movement_label in [1, 2, 9, 27] - extended
                digit = 2
            new_label += digit * (10 ** 0)

            # Index flex/rest/ext --> 0/1/2 (2nd digit)
            if movement_label in [3, 4, 5, 6, 8, 15, 16, 21]:  # flexed
                digit = 0
            elif movement_label in [9, 10, 11, 12, 13, 14, 17, 18, 19, 20, 22, 23, 24, 25, 27]:  # rest
                digit = 1
            else:  # movement_label in [1, 2, 7, 26] - extended
                digit = 2
            new_label += digit * (10 ** 1)

            # Middle flex/rest/ext --> 0/1/2 (3rd digit)
            if movement_label in [3, 4, 5, 6, 8, 15, 16, 20]:  # flexed
                digit = 0
            elif movement_label in [9, 10, 11, 12, 13, 14, 17, 18, 19, 21, 22, 23, 24, 26, 27]:  # rest
                digit = 1
            else:  # movement_label in [1, 2, 7, 25] - extended
                digit = 2
            new_label += digit * (10 ** 2)

            # Ring flex/rest/ext --> 0/1/2 (4th digit)
            if movement_label in [3, 4, 5, 6, 8, 15, 16, 19]:  # flexed
                digit = 0
            elif movement_label in [9, 10, 11, 12, 13, 14, 17, 18, 20, 21, 22, 23, 25, 26, 27]:  # rest
                digit = 1
            else:  # movement_label in [1, 2, 7, 24] - extended
                digit = 2
            new_label += digit * (10 ** 3)

            # Pinky flex/rest/ext --> 0/1/2 (5th digit)
            if movement_label in [3, 4, 5, 6, 8, 15, 16, 18]:  # flexed
                digit = 0
            elif movement_label in [9, 10, 11, 12, 13, 14, 17, 19, 20, 21, 22, 24, 25, 26, 27]:  # rest
                digit = 1
            else:  # movement_label in [1, 2, 7, 23] - extended
                digit = 2
            new_label += digit * (10 ** 4)

            # Supination codes --> 0/1/2 (6th digit)
            if baseline_label == 1:  # palm/fist up
                digit = 0
            elif baseline_label == 2:  # palm/fist side
                digit = 1
            else:  # baseline_label == 3 - palm/fist down
                digit = 2
            new_label += digit * (10 ** 5)

            # Wrist codes (7th digit)
            if movement_label in [13, 14]:
                digit = 0  # Up(-90)
            elif movement_label in [11, 12]:
                digit = 2  # Down(90)
            else:  # Straight(0)
                digit = 1
            new_label += digit * (10 ** 6)

            # Thumb abduction (8th digit)
            if movement_label in [1, 2, 9, 27]:  # dorsal
                digit = 0
            elif movement_label in [7, 8, 11, 12, 13, 14, 18, 19, 20, 21, 22, 23, 24, 25, 26]:  # rest
                digit = 1
            else:  # movement_label in [3, 4, 5, 6, 10, 15, 16, 17, 22] - palmar
                digit = 2
            new_label += digit * (10 ** 7)

            # Strength (9th digit)
            if movement_label in [5, 12, 13]:  # normal
                digit = 2
            elif movement_label in [6, 11, 14]:  # high
                digit = 3
            elif movement_label in []:  # low
                digit = 1
            else:  # not specified
                digit = 0
            new_label += digit * (10 ** 8)

            # Speed (10th digit)
            if movement_label in [1, 3]:  # low
                digit = 1
            elif movement_label in [2, 4]:  # high
                digit = 3
            elif movement_label in []:  # normal
                digit = 2
            else:  # not specified
                digit = 0
            new_label += digit * (10 ** 9)

        new_labels[i] = new_label
    
    return new_labels

def convert_labels_to_dof_dict(y, strength_and_speed: bool):
    """
    Convert the final label array y to a dictionary of 8 degrees of freedom.
    
    Assumes y contains the 8-digit encoded labels from your original mapping.
    
    Returns: dict with keys 'dof_1' to 'dof_8' representing each degree of freedom
    """
    # Initialize arrays for each degree of freedom
    dof_1 = np.full(len(y), -1, dtype=int)  # Thumb flexion
    dof_2 = np.full(len(y), -1, dtype=int)  # Index flexion  
    dof_3 = np.full(len(y), -1, dtype=int)  # Middle flexion
    dof_4 = np.full(len(y), -1, dtype=int)  # Ring flexion
    dof_5 = np.full(len(y), -1, dtype=int)  # Little flexion
    dof_6 = np.full(len(y), -1, dtype=int)  # Supination
    dof_7 = np.full(len(y), -1, dtype=int)  # Wrist angle
    dof_8 = np.full(len(y), -1, dtype=int)  # Thumb abduction
    dof_9 = np.full(len(y), -1, dtype=int)  # Strength
    dof_10 = np.full(len(y), -1, dtype=int)  # Speed
    
    for i, label in enumerate(y):
        if label == -1:
            # Skip invalid labels
            continue
            
        # Decode the 8-digit number back to individual DoFs
        # Assuming the encoding is: dof_1 * 10^0 + dof_2 * 10^1 + ... + dof_8 * 10^7
        temp_label = label
        dof_1[i] = temp_label % 10
        temp_label //= 10
        dof_2[i] = temp_label % 10
        temp_label //= 10
        dof_3[i] = temp_label % 10
        temp_label //= 10
        dof_4[i] = temp_label % 10
        temp_label //= 10
        dof_5[i] = temp_label % 10
        temp_label //= 10
        dof_6[i] = temp_label % 10
        temp_label //= 10
        dof_7[i] = temp_label % 10
        temp_label //= 10
        dof_8[i] = temp_label % 10
        if strength_and_speed:
            temp_label //= 10
            dof_9[i] = temp_label % 10
            temp_label //= 10
            dof_10[i] = temp_label % 10
    if strength_and_speed:
        return {
            'dof_1': dof_1,  # Thumb flexion (0: flexed, 1: rest, 2: extended)
            'dof_2': dof_2,  # Index flexion (0: flexed, 1: rest, 2: extended)
            'dof_3': dof_3,  # Middle flexion (0: flexed, 1: rest, 2: extended)
            'dof_4': dof_4,  # Ring flexion (0: flexed, 1: rest, 2: extended)
            'dof_5': dof_5,  # Little flexion (0: flexed, 1: rest, 2: extended)
            'dof_6': dof_6,  # Supination (0: up, 1: side, 2: down)
            'dof_7': dof_7,  # Wrist angle (0: up, 1: straight, 2: down)
            'dof_8': dof_8,  # Thumb abduction (0: dorsal, 1: rest, 2: palmar)
            'dof_9': dof_9,  # Strength (0: NaN, 0: Low, 1: Medium, 2: High)
            'dof_10': dof_10,  # Speed (0: NaN, 0: Low, 1: Medium, 2: High)
            'original_labels': y  # Keep the original encoded labels
        }
    else:
        return {
            'dof_1': dof_1,  # Thumb flexion (0: flexed, 1: rest, 2: extended)
            'dof_2': dof_2,  # Index flexion (0: flexed, 1: rest, 2: extended)
            'dof_3': dof_3,  # Middle flexion (0: flexed, 1: rest, 2: extended)
            'dof_4': dof_4,  # Ring flexion (0: flexed, 1: rest, 2: extended)
            'dof_5': dof_5,  # Little flexion (0: flexed, 1: rest, 2: extended)
            'dof_6': dof_6,  # Supination (0: up, 1: side, 2: down)
            'dof_7': dof_7,  # Wrist angle (0: up, 1: straight, 2: down)
            'dof_8': dof_8,  # Thumb abduction (0: dorsal, 1: rest, 2: palmar)
            'original_labels': y  # Keep the original encoded labels
        }

In [12]:
def get_emg_labels_from_path(repo_root, subject="P005", session="S002", task="Default", run="001_eeg_up"):
    """
    Load EMG data and labels from a BIDS-compliant directory.
    Returns: X (samples x channels), y (labels)
    """
    streams, header = load_emg_bids(repo_root, subject=subject, session=session, task=task, run=run)

    emg_channels = get_emg_channels(streams)
    trigger_labels = streams[0]

    emg_labeled = map_triggers_to_labels(emg_channels, trigger_labels)

    return emg_labeled


def load_single_emg_file(repo_root, strength_and_speed=False, subject="P005", session="S002", task="Default", run="001_eeg_up"):
    """
    Load EMG data and labels from a BIDS-compliant directory.
    """
    try:
        print(f"Loading EMG data for subject {subject}, session {session}, task {task}, run {run}")
        
        print("Step 1: Loading BIDS data...")
        streams, header = load_emg_bids(repo_root, subject=subject, session=session, task=task, run=run)
        
        if not streams:
            raise ValueError(f"No EMG data found for the specified parameters")
        
        print("Step 2: Getting EMG channels...")
        emg_channels = get_emg_channels(streams)
        
        print("Step 3: Processing triggers and labels...")
        trigger_stream = streams[0]
        labeled_data = map_triggers_to_labels(emg_channels, trigger_stream)

        print("Step 4: Converting to arrays...")
        # Try with explicit dtype conversion
        try:
            X_raw = np.array(labeled_data['time_series'], dtype=np.float32)
        except OverflowError:
            print("Warning: Overflow in time_series, trying float64")
            X_raw = np.array(labeled_data['time_series'], dtype=np.float64)
        
        try:
            y_raw = np.array(labeled_data['labels'], dtype=np.int32)
        except OverflowError:
            print("Warning: Overflow in labels, trying int64")
            y_raw = np.array(labeled_data['labels'], dtype=np.int64)
            
        timestamps = np.array(labeled_data['time_stamps'], dtype=np.float64)

        print(f"Successfully loaded.")

        y_raw_dict = convert_labels_to_dof_dict(y_raw, strength_and_speed)

        data_dict = {
            'X': X_raw,
            'y': y_raw_dict,
            't': timestamps,
            'sub': subject,
            'ses': session,
            'task': task,
            'run': run
        }
    
        return data_dict
    
    except Exception as e:
        print(f"Error loading EMG data: {e}")
        import traceback
        traceback.print_exc()
        return None
    

def load_emg_data(repo_root, strength_and_speed=False, **filters):
    """
    Loads all EMG data files from a BIDS-compliant dataset that match
    the provided filters.

    Args:
        repo_root (str): Root path to BIDS dataset.
        strength_and_speed (bool): Include DoF describing strength and speed of the movement
        **filters: Keyword arguments to filter by.
            Keys must be BIDS components: 'subject', 'session', 'task', 'run'.
            Values can be a single string or a list of strings.
            If a key is omitted, all values for that component are loaded.

    Example:
        # Load all data
        all_data = load_emg_data(repo_root)
        
        # Load specific subjects
        data_p1_p2 = load_emg_data(repo_root, subject=['P001', 'P002'])
        
        # Load one subject, one session
        data_p1_s1 = load_emg_data(repo_root, subject='P001', session='S001')
        
        # Load two runs for all subjects
        data_runs = load_emg_data(repo_root, run=['001_eeg_up', '002_eeg_down'])
    """
    
    # 1. Normalize filters: Ensure all filter values are lists
    processed_filters = {}
    for key, value in filters.items():
        if isinstance(value, str):
            processed_filters[key] = [value] # Convert single string to list
        elif value is not None:
            processed_filters[key] = value # Already a list
            
    print(f"Starting data load with filters: {processed_filters}")

    # 2. Compile BIDS regex to parse filenames
    # This pattern captures sub, ses, task, and run from a path
    # bids_pattern = re.compile(
    #     r'sub-([a-zA-Z0-9]+)'         # Group 1: subject
    #     r'(?:_ses-([a-zA-Z0-9]+))?'   # Group 2: session (optional)
    #     r'(?:_task-([a-zA-Z0-9]+))?'  # Group 3: task (optional)
    #     r'(?:_run-([a-zA-Z0-9_]+))?'  # Group 4: run (optional)
    # )
    bids_pattern = re.compile(
        r'sub-([a-zA-Z0-9]+)'           # Group 1: subject
        r'(?:_ses-([a-zA-Z0-9]+))?'     # Group 2: session (optional)
        r'(?:_task-([a-zA-Z0-9]+))?'    # Group 3: task (optional)  
        r'(?:_run-([a-zA-Z0-9_\-]+))?'  # Group 4: run (optional) - now allows underscores AND hyphens
    )

    # 3. Walk the directory and find all files that match the BIDS pattern
    discovered_files = []
    for root, dirs, files in os.walk(repo_root):
        for file in files:
            # Look for common data file extensions
            if file.endswith(('.vhdr', '.eeg', '.vmrk', '.edf', '.bdf', '.xdf', '.gdf', '.set')):
                print(os.path.join(root, file))
                match = bids_pattern.search(file) #os.path.join(root, file)
                if match:
                    discovered_files.append(match)

    if not discovered_files:
        print("Warning: No BIDS-compatible files found in repo_root.")
        return []

    # 4. Filter the discovered files
    all_data_dicts = []
    loaded_file_keys = set() # Avoid loading .vhdr, .eeg, .vmrk as 3 separate files

    for match in discovered_files:
        # Create a unique key for this file's components
        file_key = match.group(0) # e.g., "sub-P005_ses-S002_task-Default_run-001_eeg_up"
        if file_key in loaded_file_keys:
            continue
        
        components = {
            'subject': match.group(1),
            'session': match.group(2), # Will be None if not found
            'task': match.group(3),    # Will be None if not found
            'run': match.group(4)      # Will be None if not found
        }
        
        # Check this file against the user's filters
        keep_file = True
        for key, allowed_values in processed_filters.items():
            file_value = components.get(key)
            if file_value not in allowed_values:
                keep_file = False
                break # Mismatch, skip this file

        if keep_file:
            # This file matches! Add its key to avoid duplicates.
            loaded_file_keys.add(file_key)
            
            # Prepare arguments for load_single_emg_file
            # We only pass components that were *actually found* in the filename.
            # If a component is None, it's not passed, so the
            # load_single_emg_file function will use its *own* default.
            load_args = {'repo_root': repo_root, 'strength_and_speed': strength_and_speed}
            for key, val in components.items():
                if val is not None:
                    load_args[key] = val
                    
            # 5. Load the matching file
            data_dict = load_single_emg_file(**load_args)
            
            if data_dict:
                all_data_dicts.append(data_dict)

    print(f"\nLoad complete. Successfully loaded {len(all_data_dicts)} files.")
    return all_data_dicts

In [13]:
dict_loaded = load_emg_data(repo_root, subject="P005", session="S002", task="Default", run="001_eeg_up")

Starting data load with filters: {'subject': ['P005'], 'session': ['S002'], 'task': ['Default'], 'run': ['001_eeg_up']}
c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg\sub-01_task-Cyl_acq-cometa_emg.edf
c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-P005\ses-S002\emg\sub-P005_ses-S002_task-Default_run-001_eeg_up.xdf
Loading EMG data for subject P005, session S002, task Default, run 001_eeg_up
Step 1: Loading BIDS data...
Step 2: Getting EMG channels...
Step 3: Processing triggers and labels...
Step 4: Converting to arrays...
Successfully loaded.

Load complete. Successfully loaded 1 files.


In [14]:
dict_unique = load_single_emg_file(repo_root, strength_and_speed=False, subject="P005", session="S002", task="Default", run="001_eeg_up")

Loading EMG data for subject P005, session S002, task Default, run 001_eeg_up
Step 1: Loading BIDS data...
Step 2: Getting EMG channels...
Step 3: Processing triggers and labels...
Step 4: Converting to arrays...
Successfully loaded.


In [15]:
dict_unique

{'X': array([[  7099.552,  -3839.134, -13069.982,   4543.904,  -1796.046,
           2896.262],
        [  7100.148,  -3838.091, -13067.002,   4554.334,  -1796.344,
           2895.219],
        [  7106.406,  -3835.409, -13060.595,   4562.678,  -1797.238,
           2897.603],
        ...,
        [  4408.314,  -6320.878, -14723.137,   3769.849,    195.637,
           4496.522],
        [  4404.589,  -6321.474, -14724.478,   3730.364,    182.674,
           4497.565],
        [  4406.526,  -6318.196, -14722.541,   3704.736,    172.84 ,
           4501.886]], shape=(1773968, 6), dtype=float32),
 'y': {'dof_1': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
  'dof_2': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
  'dof_3': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
  'dof_4': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
  'dof_5': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
  'dof_6': array([-1, -1, -1, ...,  0,  0,  0], shape=(1773

In [16]:
dict_loaded

[{'X': array([[  7099.552,  -3839.134, -13069.982,   4543.904,  -1796.046,
            2896.262],
         [  7100.148,  -3838.091, -13067.002,   4554.334,  -1796.344,
            2895.219],
         [  7106.406,  -3835.409, -13060.595,   4562.678,  -1797.238,
            2897.603],
         ...,
         [  4408.314,  -6320.878, -14723.137,   3769.849,    195.637,
            4496.522],
         [  4404.589,  -6321.474, -14724.478,   3730.364,    182.674,
            4497.565],
         [  4406.526,  -6318.196, -14722.541,   3704.736,    172.84 ,
            4501.886]], shape=(1773968, 6), dtype=float32),
  'y': {'dof_1': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
   'dof_2': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
   'dof_3': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
   'dof_4': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
   'dof_5': array([-1, -1, -1, ...,  1,  1,  1], shape=(1773968,)),
   'dof_6': array([-1, -1, -1, ...,  0,  

In [17]:
dict_loaded[0]["X"].shape

(1773968, 6)

In [18]:
def segment_aux_windows_new(data, window_ms=200, step_ms=100, sampling_rate=1000):
    """
    Create windowed DataFrame directly from EMG data array.
    
    Args:
        data: numpy array of shape (n_samples, n_channels) - EMG data
        window_ms: window length in milliseconds
        step_ms: step size between windows in milliseconds  
        sampling_rate: sampling rate in Hz
    
    Returns:
        pd.DataFrame: Windowed data with columns as channel indices and window indices as rows
    """
    window_samples = int((window_ms / 1000.0) * sampling_rate)
    step_samples = int((step_ms / 1000.0) * sampling_rate)
    n_windows = (data.shape[0] - window_samples) // step_samples + 1
    
    windowed_data = {ch: [] for ch in range(len(data[0]))}
    
    for i in range(n_windows):
        start = i * step_samples
        end = start + window_samples
        if end > data.shape[0]:
            break
        for ch_idx, ch_name in enumerate(range(len(data[0]))):
            windowed_data[ch_name].append(data[start:end, ch_idx])
    
    return pd.DataFrame(windowed_data)

def notch_filter(df, fs=1000, freq=50.0, q=30.0):
    """Remove power line interference at 50 Hz (or 60 Hz for US)"""
    b, a = signal.iirnotch(freq, q, fs)
    
    filtered_df = df.copy()
    for col in df.columns:
        # Skip non-time-series columns
        if col == 'window_index':
            continue
        # Apply filter only to columns that contain arrays
        filtered_df[col] = df[col].apply(lambda x: signal.filtfilt(b, a, x) if isinstance(x, (list, np.ndarray)) and len(x) > 1 else x)
    
    return filtered_df


def passband_filter(df, fs=1000, lowcut=20.0, highcut=300.0, order=4):
    """Bandpass filter for EMG signals (20-450 Hz)"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    
    filtered_df = df.copy()
    for col in df.columns:
        # Skip non-time-series columns
        if col == 'window_index':
            continue
        # Apply filter only to columns that contain arrays
        filtered_df[col] = df[col].apply(lambda x: signal.filtfilt(b, a, x) if isinstance(x, (list, np.ndarray)) and len(x) > 1 else x)
    
    return filtered_df

In [44]:
dict_loaded[0]["X"]

array([[  7099.552,  -3839.134, -13069.982,   4543.904,  -1796.046,
          2896.262],
       [  7100.148,  -3838.091, -13067.002,   4554.334,  -1796.344,
          2895.219],
       [  7106.406,  -3835.409, -13060.595,   4562.678,  -1797.238,
          2897.603],
       ...,
       [  4408.314,  -6320.878, -14723.137,   3769.849,    195.637,
          4496.522],
       [  4404.589,  -6321.474, -14724.478,   3730.364,    182.674,
          4497.565],
       [  4406.526,  -6318.196, -14722.541,   3704.736,    172.84 ,
          4501.886]], shape=(1773968, 6), dtype=float32)

In [20]:
windowed_df = segment_aux_windows_new(dict_loaded[0]["X"])

In [21]:
windowed_df

,0,1,2,3,4,5
0,"[7099.552, 7100.148, 7106.406, 7112.962, 7128....","[-3839.134, -3838.091, -3835.409, -3829.896, -...","[-13069.982, -13067.002, -13060.595, -13052.69...","[4543.904, 4554.334, 4562.678, 4559.251, 4557....","[-1796.046, -1796.344, -1797.238, -1800.069, -...","[2896.262, 2895.219, 2897.603, 2903.414, 2906...."
1,"[7096.125, 7100.744, 7107.151, 7112.664, 7125....","[-3848.67, -3844.796, -3842.114, -3839.879, -3...","[-13073.111, -13070.429, -13060.744, -13049.27...","[4537.944, 4536.454, 4538.689, 4544.202, 4556....","[-1826.889, -1832.104, -1830.316, -1832.402, -...","[2918.761, 2920.102, 2920.549, 2924.274, 2933...."
2,"[7086.887, 7087.632, 7093.89, 7101.936, 7111.9...","[-3841.965, -3837.644, -3837.346, -3831.982, -...","[-13072.664, -13066.406, -13062.085, -13053.29...","[4541.073, 4534.368, 4532.133, 4539.434, 4542....","[-1821.376, -1825.548, -1829.124, -1827.634, -...","[2937.386, 2940.217, 2941.26, 2943.644, 2947.3..."
3,"[7087.036, 7087.334, 7090.761, 7098.807, 7109....","[-3835.856, -3830.343, -3830.045, -3824.681, -...","[-13073.26, -13070.131, -13061.34, -13054.188,...","[4540.477, 4540.924, 4543.755, 4545.394, 4550....","[-1830.614, -1831.359, -1830.763, -1830.316, -...","[2936.939, 2941.856, 2945.134, 2950.796, 2953...."
4,"[7087.036, 7088.526, 7094.784, 7102.83, 7122.3...","[-3826.767, -3825.426, -3823.936, -3817.827, -...","[-13072.068, -13063.873, -13057.615, -13045.54...","[4545.841, 4542.116, 4544.202, 4558.208, 4575....","[-1834.19, -1840.448, -1842.981, -1839.256, -1...","[2919.059, 2920.549, 2924.721, 2931.426, 2943...."
...,...,...,...,...,...,...
17733,"[4147.564, 4103.609, 4062.783, 4030.748, 3998....","[-6103.785, -6130.009, -6153.104, -6173.368, -...","[-14143.229, -14162.003, -14180.777, -14192.54...","[3945.073, 3975.32, 3953.566, 4001.842, 4020.6...","[-463.986, -463.986, -469.201, -459.665, -447....","[5106.379, 5101.909, 5099.525, 5113.531, 5127...."
17734,"[4809.124, 4779.324, 4749.822, 4728.962, 4716....","[-6485.97, -6498.784, -6513.535, -6522.773, -6...","[-14493.975, -14516.474, -14535.397, -14537.48...","[5783.286, 5840.204, 5841.992, 5841.694, 5858....","[-1.639, 24.287, 35.164, 45.594, 60.643, 78.37...","[5778.667, 5761.979, 5745.589, 5736.649, 5732...."
17735,"[5202.186, 5180.879, 5164.787, 5156.294, 5148....","[-6254.573, -6278.264, -6290.78, -6301.21, -63...","[-14551.638, -14565.644, -14578.011, -14580.24...","[5394.992, 5400.356, 5404.379, 5402.442, 5386....","[-71.222, -79.566, -71.222, -72.414, -98.787, ...","[4793.628, 4773.513, 4762.04, 4753.845, 4742.8..."
17736,"[4891.968, 4870.512, 4854.42, 4848.311, 4846.0...","[-6324.007, -6347.251, -6362.3, -6377.349, -63...","[-14594.997, -14615.261, -14634.929, -14640.29...","[4783.943, 4789.009, 4796.906, 4799.141, 4817....","[170.158, 167.029, 192.359, 190.869, 198.915, ...","[5039.031, 5021.151, 5019.661, 5024.727, 5036...."


In [22]:
windowed_df.shape

(17738, 6)

In [23]:
preproc_df = notch_filter(windowed_df)
preproc_df = passband_filter(preproc_df) 

In [24]:
preproc_df

,0,1,2,3,4,5
0,"[-1.4950598477541401, -2.9635871781445817, -3....","[-1.3638874513155552, -2.7016718854663093, -3....","[-1.2328368993303194, -0.9577662006699881, 1.1...","[0.07956095555389987, 9.647539947136051, 14.23...","[1.9551805909413171, 3.537335348687557, 7.3287...","[-1.3163536849762378, -5.1313056456631, -5.584..."
1,"[-0.7438142887086721, 1.6976021515029984, 3.05...","[-1.2580578051447318, 0.6685436562829981, -0.5...","[-0.7234575034216539, -0.7420242203172247, 3.6...","[-0.36205397181016274, -2.9667453296487496, -4...","[1.4605693766976184, -0.317470183601249, 1.096...","[-1.5801991889096387, -2.870905554155904, -4.0..."
2,"[-1.4390597693274128, -2.8552522171093755, -1....","[-1.1785797820186892, 0.05117639756287984, -0....","[-1.078116590104376, 1.3241671189360806, 2.216...","[-0.1116141412891678, -8.448717442438848, -9.5...","[1.946996240881921, -0.8924737625880241, 1.474...","[-1.277783295509136, -1.3660201173348427, -2.8..."
3,"[-1.1445016943261987, -3.4346440613324054, -4....","[-1.300961778691577, 0.4355390988606318, 0.320...","[-0.9906690527839701, -0.415834056456029, 2.39...","[-1.3353025343170326, -1.6850778790894845, -1....","[1.1759288110744435, 2.499336723158204, 6.0962...","[-1.3515311079140127, -0.29897167890399534, 0...."
4,"[-1.0340104136472958, -2.311603106990687, -2.1...","[-0.6196742544849829, -1.8028463171722364, -2....","[-0.9743187009076255, 2.6793826853546183, 6.71...","[-0.3857839574778583, -7.241413911365918, -5.3...","[1.656998841404856, -3.14783946517164, -1.5110...","[-0.9715469829848176, -2.9855420541720328, -2...."
...,...,...,...,...,...,...
17733,"[-2.011313380091468, -3.179427670485718, 1.691...","[-0.6501540156770997, -13.144358290396887, -24...","[-0.7582022400846015, -10.356533275384132, -21...","[-4.952114515240881, 1.3123995247138467, 21.18...","[-3.3113040259073174, -10.509309523356999, -10...","[1.8312252989023674, -11.000640248772124, -16...."
17734,"[2.8378994634655834, -17.488784301155725, -35....","[2.8972393018204414, -8.350171240725938, -20.0...","[0.8204402698263982, -18.596429264989766, -32....","[-1.7387564308878622, 44.19509954562362, 47.77...","[-2.56374293812108, 7.7881941944965565, 7.8398...","[-3.157025768135945, -8.80310344287286, -14.25..."
17735,"[-0.660705576860694, -14.462651356312382, -22....","[1.2836005721696377, -13.337795826030067, -22....","[0.5256796787023809, -8.3125411677557, -17.714...","[-1.0522372855623134, 8.364642450181092, 16.21...","[0.5148716990069131, -3.495822902578928, 3.799...","[-0.010183618711068099, -8.419594983251642, -1..."
17736,"[1.7876167912314274, -12.573040731742605, -21....","[1.7422309465866825, -12.272801249891945, -24....","[-0.0237139911531159, -17.382486895435434, -34...","[5.563167545697727, 19.876810540964506, 33.888...","[0.12388384083763154, -7.807917978457016, -9.3...","[1.4548552931067809, -16.926762555344354, -26...."


In [25]:
preproc_df.shape

(17738, 6)

In [39]:
from scipy.fft import fft, fftfreq
# ============= Time Domain Features =============

def mav(x):
    """Mean Absolute Value"""
    return np.mean(np.abs(x))

def var(x):
    """Variance"""
    return np.var(x)

def rms(x):
    """Root Mean Square"""
    return np.sqrt(np.mean(x**2))

def ssc(x, threshold=0):
    """Slope Sign Change"""
    diff = np.diff(x)
    sign_changes = np.sum(np.diff(np.sign(diff)) != 0)
    return sign_changes

def zc(x, threshold=0):
    """Zero Crossing"""
    return np.sum(np.diff(np.sign(x)) != 0)

def wl(x):
    """Wave Length"""
    return np.sum(np.abs(np.diff(x)))

def ar5(x):
    """5th order Autoregressive coefficients"""
    # Burg method for AR coefficients
    r = np.correlate(x, x, mode='full')[len(x)-1:]
    r = r[:6] / r[0]
    
    # Levinson-Durbin recursion
    ar = np.zeros(5)
    e = r[0]
    for k in range(5):
        lambda_k = r[k+1] - np.sum(ar[:k] * r[k:0:-1])
        ar_new = np.zeros(k+2)
        ar_new[k+1] = lambda_k / e
        ar_new[:k+1] = ar[:k+1] - ar_new[k+1] * ar[k::-1]
        ar = ar_new
        e = e * (1 - ar_new[k+1]**2)
    
    return ar[1:]

def cc(x, n_coeff=5):
    """Cepstral Coefficients"""
    spectrum = np.abs(fft(x))
    log_spectrum = np.log(spectrum[:len(spectrum)//2] + 1e-10)
    cepstrum = np.real(fft(log_spectrum))
    return cepstrum[:n_coeff]


# ============= Frequency Domain Features =============

def mnf(x, fs=1000):
    """Mean Frequency"""
    freqs = fftfreq(len(x), 1/fs)
    psd = np.abs(fft(x))**2
    
    # Only positive frequencies
    idx = freqs > 0
    freqs = freqs[idx]
    psd = psd[idx]
    
    return np.sum(freqs * psd) / np.sum(psd)

def mdf(x, fs=1000):
    """Median Frequency"""
    freqs = fftfreq(len(x), 1/fs)
    psd = np.abs(fft(x))**2
    
    # Only positive frequencies
    idx = freqs > 0
    freqs = freqs[idx]
    psd = psd[idx]
    
    cumsum = np.cumsum(psd)
    median_idx = np.where(cumsum >= cumsum[-1] / 2)[0][0]
    return freqs[median_idx]


# ============= Time-Frequency Features =============

def wtwl(x, wavelet='db4', level=4):
    """Wavelet Transform Waveform Length"""
    coeffs = pywt.wavedec(x, wavelet, level=level)
    detail = coeffs[-1]  # Last detail coefficient
    return np.sum(np.abs(np.diff(detail)))

def wtvar(x, wavelet='db4', level=4):
    """Wavelet Transform Variance"""
    coeffs = pywt.wavedec(x, wavelet, level=level)
    detail = coeffs[-1]
    return np.var(detail)

def wtmav(x, wavelet='db4', level=4):
    """Wavelet Transform Mean Absolute Value"""
    coeffs = pywt.wavedec(x, wavelet, level=level)
    detail = coeffs[-1]
    return np.mean(np.abs(detail))

In [40]:
def extract_window_features(window, fs=1000):
    """Extract all features from a single window"""
    features = {}
    
    # Time domain
    features['MAV'] = mav(window)
    features['VAR'] = var(window)
    features['RMS'] = rms(window)
    features['SSC'] = ssc(window)
    features['ZC'] = zc(window)
    features['WL'] = wl(window)
    
    # AR coefficients
    ar_coefs = ar5(window)
    for i, coef in enumerate(ar_coefs):
        features[f'AR{i+1}'] = coef
    
    # Cepstral coefficients
    cc_coefs = cc(window)
    for i, coef in enumerate(cc_coefs):
        features[f'CC{i+1}'] = coef
    
    # Frequency domain
    features['MNF'] = mnf(window, fs)
    features['MDF'] = mdf(window, fs)
    
    # Time-frequency
    features['WTWL'] = wtwl(window)
    features['WTVAR'] = wtvar(window)
    features['WTMAV'] = wtmav(window)
    
    return features

In [41]:
features_list = []
for idx in range(len(preproc_df)):
    window_features = {}
    
    for ch in preproc_df.columns:
        signal_window = preproc_df.loc[idx, ch]
        ch_features = extract_window_features(signal_window, fs=1000)
        
        # Prefix with channel name
        for feat_name, feat_val in ch_features.items():
            window_features[f"{ch}_{feat_name}"] = feat_val
    
    features_list.append(window_features)

# Create DataFrame
features_df = pd.DataFrame(features_list)

In [42]:
features_df.shape

(17738, 126)

In [43]:
features_df

,0_MAV,0_VAR,0_RMS,0_SSC,0_ZC,0_WL,0_AR1,0_AR2,0_AR3,0_AR4,...,5_CC1,5_CC2,5_CC3,5_CC4,5_CC5,5_MNF,5_MDF,5_WTWL,5_WTVAR,5_WTMAV
0,15.336947,286.904216,16.977729,51,21,921.616086,-0.017257,-0.684484,-0.144869,0.928598,...,261.853176,-20.826391,4.926609,-12.104876,-1.035889,82.453042,50.0,245.323147,2.633885,1.309019
1,15.518328,289.543062,17.061940,46,21,910.324769,0.038824,-0.650330,-0.196541,0.880909,...,268.874966,-8.547848,8.936096,-4.747193,0.987352,81.435028,50.0,221.200775,2.326071,1.208611
2,15.252026,281.985799,16.822077,49,21,957.011371,-0.043447,-0.710012,-0.124019,0.954726,...,280.856155,-16.942431,6.501405,-9.273604,-0.820192,86.710525,50.0,287.330223,3.819123,1.491593
3,15.541981,298.007995,17.310925,44,19,996.559392,0.005791,-0.661018,-0.162589,0.905797,...,269.618876,-22.053050,17.147705,-3.031373,-4.556904,85.853068,50.0,296.602508,4.136802,1.534909
4,15.504349,293.163817,17.178819,49,19,953.851959,0.037173,-0.640617,-0.191216,0.876933,...,152.613191,-76.271935,18.027695,3.131820,0.321642,77.172563,50.0,260.999730,3.018497,1.366539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17733,24.309867,1165.202822,34.160600,43,18,1460.919627,0.486477,-0.297105,-0.567475,0.462839,...,325.360940,-15.987620,6.029126,-11.949506,-10.879703,68.907230,50.0,383.294999,6.782371,2.052275
17734,15.825080,374.821464,19.456861,55,25,1174.904607,0.165038,-0.487762,-0.282357,0.761308,...,357.423414,1.372378,18.141611,-5.960160,3.364721,68.197264,50.0,396.592270,7.672911,2.192261
17735,14.306551,273.095279,16.526616,54,19,1164.536811,-0.283871,-0.823310,0.110981,1.179617,...,288.511788,-50.869087,3.414076,-18.911175,-8.700611,77.946844,50.0,433.264245,9.232698,2.379367
17736,14.013814,276.094665,16.616928,54,23,1168.864969,-0.379740,-0.903815,0.194756,1.261468,...,372.200291,3.844141,6.411317,-4.474342,4.844662,74.829013,50.0,423.930339,8.743999,2.293502
